# Playground

Ambiente configurável para gerar cenários de grafos, acoplar algoritmos e comparar todos sobre **as mesmas instâncias**. Ajuste as constantes da seção 1 e rode o notebook inteiro.

Garantias do playground: todos os algoritmos recebem a mesma instância (conferida pelo hash), o mesmo tempo-limite e sementes derivadas de forma determinística; o ótimo de referência é calculado uma vez por instância; o fluxo é sempre recalculado sobre os arcos interditados.

## 1. Imports, instâncias e constantes

In [ ]:
import numpy as np
import pandas as pd

from qmfnip.evaluation import comparison_table, frontier, summarize
from qmfnip.instances import Scenario, max_flow
from qmfnip.outputs import ExperimentOutput, latest_experiment
from qmfnip.playground import ExperimentConfig, run_experiment
from qmfnip.plots import plot_frontier, plot_metric
from qmfnip.solvers import SOLVERS, MILPSolver, SolveResult, Solver

Constantes do cenário e da avaliação:

- `FAMILY`: `"random"` (rede aleatória esparsa, `SIZES` = número de nós) ou `"layered"` (rede em camadas, `SIZES` = largura k; a rede tem k×k+2 nós).
- `BUDGET_FRACTIONS`: orçamento como fração do custo do corte que zera o fluxo.
- `REPETITIONS`: execuções por par (instância, algoritmo).
- `TIME_LIMIT`: orçamento de tempo T, igual para todos os algoritmos.
- `REFERENCE_TIME_LIMIT`: tempo do MILP que calcula o ótimo de referência.
- `Z` e `P`: gap aceito e fração mínima de sucesso (ver `docs/protocolo.md`). Os valores abaixo são provisórios até o protocolo ser congelado.

In [ ]:
FAMILY = "layered"
SIZES = (3, 4, 6, 8)
BUDGET_FRACTIONS = (0.25, 0.5, 0.75)
SEEDS = (0, 1, 2)
REPETITIONS = 3
TIME_LIMIT = 10.0
REFERENCE_TIME_LIMIT = 120.0
BASE_SEED = 0
Z = 0.05
P = 0.9

Exemplo de algoritmo acoplado: uma linha de base que interdita arcos em ordem aleatória até esgotar o orçamento. Qualquer classe que herde de `Solver` e implemente `solve` pode entrar na lista `solvers`. Para usar o mesmo algoritmo com parâmetros diferentes, passe `label="..."` a cada instância dele.

In [ ]:
class RandomInterdictionSolver(Solver):
    name = "Random interdiction within budget"
    short_name = "random_baseline"

    def solve(self, instance, seed, time_limit):
        rng = np.random.default_rng(seed)
        interdicted = np.zeros(instance.m, dtype=bool)
        spent = 0
        for arc in rng.permutation(instance.m):
            if spent + instance.cost[arc] <= instance.budget:
                interdicted[arc] = True
                spent += instance.cost[arc]
        return SolveResult(interdicted, "feasible", float(max_flow(instance, interdicted)))

In [ ]:
scenario = Scenario(FAMILY, SIZES, BUDGET_FRACTIONS, SEEDS)
solvers = [MILPSolver(), RandomInterdictionSolver()]
config = ExperimentConfig(
    scenario=scenario,
    solvers=solvers,
    repetitions=REPETITIONS,
    time_limit=TIME_LIMIT,
    base_seed=BASE_SEED,
    reference=MILPSolver(),
    reference_time_limit=REFERENCE_TIME_LIMIT,
)
print(f"registered solvers: {sorted(SOLVERS)}")
pd.DataFrame(
    [
        {
            "instance_id": instance.instance_id,
            "nodes": instance.n,
            "arcs": instance.m,
            "budget": instance.budget,
            "max_flow": max_flow(instance),
            "content_hash": instance.content_hash,
        }
        for instance in scenario.instances()
    ]
)

## 2. Execução dos algoritmos

Roda todas as combinações instância × algoritmo × repetição e grava tudo em `outputs/`. Para retomar uma execução interrompida, use `run_experiment(config, resume=<pasta>)`.

In [ ]:
output = run_experiment(config)
output.path

## 3. Avaliação dos resultados

Para reavaliar uma execução antiga sem rodar de novo, troque `output` por `ExperimentOutput(<pasta>)` ou `latest_experiment()`.

In [ ]:
runs = output.load_runs()
summary = summarize(runs, z=Z)
output.write_table("summary", summary)
summary

Fronteira n\*: maior tamanho em que cada algoritmo atinge gap ≤ Z em pelo menos P das execuções, dentro de T. `reached_largest = True` indica que o maior tamanho testado ainda passou, ou seja, n\* é só um limite inferior.

In [ ]:
frontier_table = frontier(summary, p=P)
output.write_json("frontier.json", frontier_table.to_dict(orient="records"))
frontier_table

In [ ]:
comparison_table(summary, "median_gap")

In [ ]:
plot_metric(summary, "median_gap", path=output.figure_path("median_gap_vs_nodes"));

In [ ]:
plot_metric(summary, "median_wall_time_s", log_y=True, path=output.figure_path("wall_time_vs_nodes"));

In [ ]:
plot_frontier(frontier_table, path=output.figure_path("frontier"));